# EDA: Шаблон анализа группы признаков

Хакатон ChemAI: Predict the Cure.
В этом ноутбуке выполняется разведочный анализ данных для одной из групп молекулярных дескрипторов.

Объединяет чек-лист команды Jazz и лучшие практики из примера по анализу корреляций.

Перед началом работы скопируйте ноутбук под именем `eda_group<номер>_<имя>.ipynb` и замените список `FEATURES` на свои признаки.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.feature_selection import VarianceThreshold
import warnings

warnings.filterwarnings('ignore')
plt.style.use('ggplot')
sns.set_palette('viridis')
%matplotlib inline

SEED = 42
np.random.seed(SEED)


## Загрузка данных

Файлы `train.csv`, `test.csv` должны находиться в папке `data/`.


In [ ]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

# Переименовываем таргеты для удобства (в исходном train они с единицами измерения)
train.rename(columns={'IC50, mM': 'IC50', 'CC50, mM': 'CC50'}, inplace=True)

train.head()


## Первичный обзор

Общая информация о данных и статистики.


In [ ]:
train.info()


In [ ]:
train.describe().round(3)


## Анализ целевых переменных

Распределения `IC50`, `CC50`, `SI` и их логарифмов (log1p).


In [ ]:
targets = ['IC50', 'CC50', 'SI']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, col in enumerate(targets):
    sns.histplot(train[col], bins=50, ax=axes[0, i], kde=True)
    axes[0, i].set_title(f'{col} (линейная шкала)')
    sns.histplot(np.log1p(train[col]), bins=50, ax=axes[1, i], kde=True)
    axes[1, i].set_title(f'log1p({col})')
plt.tight_layout()
plt.show()


Вывод: все три таргета имеют правостороннюю асимметрию. Для моделирования рекомендуется применять логарифмирование (log1p).


## Вспомогательные функции для анализа корреляций

Эти функции интерпретируют силу связи и статистическую значимость, а также вычисляют корреляцию Пирсона и Спирмена.


In [ ]:
def interpret_correlation_strength(corr_coef):
    """Интерпретирует силу коэффициента корреляции."""
    abs_corr = abs(corr_coef)
    if abs_corr < 0.1:
        return "очень слабая"
    elif abs_corr < 0.3:
        return "слабая"
    elif abs_corr < 0.5:
        return "умеренная"
    elif abs_corr < 0.7:
        return "сильная"
    else:
        return "очень сильная"

def interpret_statistical_significance(p_value):
    """Интерпретирует статистическую значимость p-value."""
    if p_value < 0.001:
        return "высокозначимо"
    elif p_value < 0.01:
        return "очень значимо"
    elif p_value < 0.05:
        return "значимо"
    else:
        return "не значимо"

def analyze_corr(df, col1, col2, show_spearman=True):
    """Анализ корреляции между двумя колонками датафрейма."""
    data = df[[col1, col2]].dropna()
    if len(data) < 3:
        print(f"Недостаточно данных для {col1} vs {col2}")
        return
    pearson_corr, pearson_p = stats.pearsonr(data[col1], data[col2])
    spearman_corr, spearman_p = stats.spearmanr(data[col1], data[col2])

    print(f"=== {col1} vs {col2} ===")
    print(f"Наблюдений: {len(data)}")
    print(f"Пирсон: {pearson_corr:.4f} ({interpret_correlation_strength(pearson_corr)})")
    print(f"p-value: {pearson_p:.6f} [{interpret_statistical_significance(pearson_p)}]")
    print(f"R^2: {pearson_corr**2:.4f} ({pearson_corr**2*100:.1f}% дисперсии)")

    if show_spearman:
        print(f"Спирмен: {spearman_corr:.4f} ({interpret_correlation_strength(spearman_corr)})")
        print(f"p-value: {spearman_p:.6f} [{interpret_statistical_significance(spearman_p)}]")
    print()


## Признаки группы

Замените список `FEATURES` на признаки вашей группы. Сейчас приведён пример для группы 1 (общие молекулярные свойства).


In [ ]:
GROUP_NAME = 'Общие молекулярные свойства'
FEATURES = [
    'MolWt', 'ExactMolWt', 'HeavyAtomMolWt', 'NumValenceElectrons',
    'NumRadicalElectrons', 'MolLogP', 'MolMR', 'TPSA', 'LabuteASA',
    'HeavyAtomCount', 'NHOHCount', 'NOCount', 'NumHAcceptors', 'NumHDonors',
    'NumHeteroatoms', 'NumRotatableBonds', 'RingCount',
    'NumAliphaticCarbocycles', 'NumAliphaticHeterocycles', 'NumAliphaticRings',
    'NumAromaticCarbocycles', 'NumAromaticHeterocycles', 'NumAromaticRings',
    'NumSaturatedCarbocycles', 'NumSaturatedHeterocycles', 'NumSaturatedRings',
    'FractionCSP3'
]
TARGETS = ['IC50', 'CC50', 'SI']


## Анализ пропусков

Проверка наличия пропущенных значений в выбранной группе признаков.


In [ ]:
missing_percent = (train[FEATURES].isnull().sum() / len(train)) * 100
missing_df = pd.DataFrame({
    'column': missing_percent.index,
    'missing_%': missing_percent.values,
    'missing_abs': train[FEATURES].isnull().sum().values
})
missing_df = missing_df.sort_values('missing_%', ascending=False)
missing_df = missing_df[missing_df['missing_%'] > 0]

if len(missing_df) == 0:
    print('Пропуски в группе отсутствуют.')
else:
    print('Пропущенные значения в группе:')
    print(missing_df)


## Анализ распределений

Гистограммы и boxplot для каждого признака группы.


In [ ]:
n_cols = 4
n_rows = int(np.ceil(len(FEATURES) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3 * n_rows))
axes = axes.flatten()
for i, feat in enumerate(FEATURES):
    ax = axes[i]
    sns.histplot(train[feat], kde=True, ax=ax)
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel('')
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3 * n_rows))
axes = axes.flatten()
for i, feat in enumerate(FEATURES):
    ax = axes[i]
    sns.boxplot(y=train[feat], ax=ax)
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel('')
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()


## Корреляционный анализ

Тепловая карта корреляций внутри группы и с целевыми переменными. Топ-5 признаков по корреляции с каждым таргетом.


In [ ]:
corr_cols = FEATURES + TARGETS
corr = train[corr_cols].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title(f'Корреляция признаков группы {GROUP_NAME}')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
for target in TARGETS:
    corr_target = corr[target].drop(FEATURES + TARGETS, errors='ignore').sort_values(ascending=False)
    print(f'\nТоп-5 корреляций с {target}:')
    print(corr_target.head(5))


При необходимости примените `analyze_corr()` для детального анализа конкретных пар.


## Поиск проблемных признаков

Выявление константных, почти константных и сильно скоррелированных признаков.


In [ ]:
# Константные (нулевая дисперсия)
constant_feats = train[FEATURES].columns[train[FEATURES].std() == 0].tolist()
print(f'Константные признаки: {constant_feats}' if constant_feats else 'Константных признаков нет.')

# Почти константные (порог дисперсии 0.01)
sel = VarianceThreshold(threshold=0.01)
sel.fit(train[FEATURES])
low_var_feats = [FEATURES[i] for i in np.where(~sel.get_support())[0]]
print(f'Почти константные (var<0.01): {low_var_feats}' if low_var_feats else 'Почти константных нет.')

# Пары с высокой корреляцией (r > 0.95)
corr_matrix = train[FEATURES].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [(col, idx, upper.loc[col, idx]) for col in upper.columns for idx in upper.index if upper.loc[col, idx] > 0.95]
if high_corr:
    print('Пары с корреляцией >0.95:')
    for pair in high_corr:
        print(f'{pair[0]} - {pair[1]}: {pair[2]:.3f}')
else:
    print('Сильно скоррелированных пар нет.')


## Выводы по группе

**Качество данных:**
- Пропуски: ...
- Выбросы: ...
- Проблемные признаки: ...

**Необходимые преобразования:**
- Масштабирование: ...
- Импутация: ...
- Удаление признаков: ...

**Наиболее коррелирующие с таргетами признаки:**
- IC50: ...
- CC50: ...
- SI: ...

**Дополнительные замечания:**


# Как отправить результат на проверку (Pull Request)

После завершения анализа в вашей копии ноутбука выполните следующие шаги.

## 1. Скопируйте ваш ноутбук в репозиторий
- Убедитесь, что ноутбук сохранён на Google Диск.
- Переименуйте его по шаблону: `eda_group<номер>_<ваше_имя>.ipynb`
- В Colab выполните:

```python
from google.colab import drive
drive.mount('/content/drive')
!cp "/content/drive/MyDrive/Colab Notebooks/eda_group1_Иван.ipynb" notebooks/
```

## 2. Создайте новую ветку
```python
!git checkout -b eda-group1-ivan
```

## 3. Закоммитьте и запушьте ветку
```python
!git add notebooks/eda_group1_Иван.ipynb
!git commit -m "EDA group 1: general molecular properties (Ivan)"
!git push origin eda-group1-ivan
```
При запросе пароля введите ваш токен GitHub.

## 4. Создайте Pull Request на GitHub
- Перейдите в репозиторий: https://github.com/NE1nthegame/jazz_team_project
- GitHub предложит создать PR из только что отправленной ветки — нажмите "Compare & pull request".
- Заголовок: например, "EDA Group 1: Общие молекулярные свойства (Иван)".
- В описании кратко укажите основные выводы.
- В правой панели Reviewers выберите одного из участников.
- Нажмите Create pull request.

## 5. Ревью и слияние
- Назначенный ревьюер просмотрит изменения и при необходимости оставит комментарии.
- Когда всё хорошо, ревьюер нажмёт Approve, а затем автор (или сам ревьюер) нажмёт Merge pull request.
- После слияния ветку можно удалить (кнопка "Delete branch").

## Важные правила
- НЕ коммитьте и не пушьте напрямую в main.
- Один Pull Request — один ноутбук (одна группа признаков).
- Перед коммитом очистите выводы ячеек: Runtime → Restart and run all и убедитесь, что всё работает.
- Проверьте, что в ноутбуке заполнена последняя Markdown-ячейка с выводами.

Если что-то пошло не так — обратитесь в общий чат.
